In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 200)
sns.set_style('whitegrid')

df = pd.read_parquet('../data/loans_clean.parquet')
print(f"Loaded {df.shape[0]:,} rows and {df.shape[1]} columns")
print(f"Default rate: {df['default'].mean():.2%}")

In [ ]:
categoricals = ['term', 'home_ownership', 'verification_status', 'purpose', 'application_type']

for col in categoricals:
    print(f"\n{'='*50}\n{col}\n{'='*50}")
    stats = df.groupby(col).agg(
        n_loans=('default', 'size'),
        default_rate=('default', 'mean')
    ).sort_values('default_rate', ascending=False).round(4)
    print(stats)

In [ ]:
numerics = ['loan_amnt', 'int_rate', 'installment', 'annual_inc', 'dti', 
            'fico_range_low', 'open_acc', 'revol_util', 'credit_history_months']

print(df[numerics].describe(percentiles=[0.01, 0.5, 0.99]).round(2))

In [ ]:
# Convert sentinel values to NaN
df['dti'] = df['dti'].replace([-1, 999], np.nan)
df['credit_history_months'] = df['credit_history_months'].replace(999, np.nan)

# Cap extreme outliers
df['annual_inc'] = df['annual_inc'].clip(upper=1_000_000)
df['revol_util'] = df['revol_util'].clip(upper=150)
df['dti'] = df['dti'].clip(upper=100)  

# Re-check the cleaned features
print(df[['annual_inc', 'dti', 'revol_util', 'credit_history_months']].describe(
    percentiles=[0.01, 0.5, 0.99]
).round(2))

In [ ]:
def default_by_buckets(df, col, n_buckets=10):
    buckets = pd.qcut(df[col], n_buckets, duplicates='drop')
    return df.groupby(buckets, observed=True).agg(
        n_loans=('default', 'size'),
        default_rate=('default', 'mean')
    ).round(4)

for col in ['fico_range_low', 'dti', 'annual_inc', 'int_rate']:
    print(f"\n{'='*60}\nDefault rate by {col} (deciles)\n{'='*60}")
    print(default_by_buckets(df, col))

In [ ]:
# Compute missingness and default rate by missingness
missing_pct = df.isnull().mean().sort_values(ascending=False)
high_missing = missing_pct[missing_pct > 0.05]  # features with >5% missing

print(f"Features with >5% missing: {len(high_missing)}")
print("\nTop missing features:")
print((high_missing * 100).round(1).to_frame('pct_missing').head(20))

In [ ]:
informative_missing_candidates = [
    'mths_since_last_delinq',
    'mths_since_last_record', 
    'mths_since_last_major_derog',
    'mths_since_recent_bc_dlq',
    'mths_since_recent_revol_delinq',
    'mths_since_recent_inq',
]

print("Default rate: missing vs present")
print("="*60)
for col in informative_missing_candidates:
    if col in df.columns:
        missing_mask = df[col].isnull()
        rate_missing = df.loc[missing_mask, 'default'].mean()
        rate_present = df.loc[~missing_mask, 'default'].mean()
        pct_missing = missing_mask.mean() * 100
        print(f"\n{col}")
        print(f"  Missing ({pct_missing:5.1f}%): default rate {rate_missing:.4f}")
        print(f"  Present ({100-pct_missing:5.1f}%): default rate {rate_present:.4f}")
        print(f"  Difference:                {rate_missing - rate_present:+.4f}")

In [ ]:
# Numeric features only (correlation needs numbers)
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
# Drop the target so we don't see correlations with default itself
numeric_cols = [c for c in numeric_cols if c not in ['default', 'issue_year']]

corr = df[numeric_cols].corr().abs()

# Get pairs above 0.9 correlation (highly redundant)
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
high_corr_pairs = (
    upper.stack()
    .sort_values(ascending=False)
    .head(20)
    .reset_index()
)
high_corr_pairs.columns = ['feature_1', 'feature_2', 'abs_correlation']
print(high_corr_pairs.round(3))